<a href="https://colab.research.google.com/github/sj-workbench/years_of_learnings/blob/main/ML_prediction_withoutpipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [66]:
# In this Titanic dataset i have worked on prediciton on the survival rate based on the inputs it takes.
# I haven't worked on the DTC yet but used in order to see how it works.
# In the OHE section i have used (drop) in order to avoid multi-collinearity but it's not necessary as i have used DTA.
# Also this dataset is not linear so multi-collinearity won't affect much.
# Without using (drop) this prediction model gives 80% accuracy but with using (drop) it gives 79% accuracy.
# IN THIS DATASET I HAVE ALSO WORKED ON THE PREDICTION OF A RANDOM INPUT'S SURVIVAL without using pipeline.

In [67]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier

In [68]:
df = pd.read_csv('/content/Titanic-Dataset.csv')
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
135,136,0,2,"Richard, Mr. Emile",male,23.0,0,0,SC/PARIS 2133,15.0458,NaN,C
262,263,0,1,"Taussig, Mr. Emil",male,52.0,1,1,110413,79.6500,E67,S
570,571,1,2,"Harris, Mr. George",male,62.0,0,0,S.W./PP 752,10.5000,NaN,S
307,308,1,1,"Penasco y Castellana, Mrs. Victor de Satode (M...",female,17.0,1,0,PC 17758,108.9000,C65,C
349,350,0,3,"Dimic, Mr. Jovan",male,42.0,0,0,315088,8.6625,NaN,S


In [69]:
df.isnull().sum()    # to check missing value
                     # columns = (age,cabin,embarked) contains missing value

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [70]:
df.drop(columns = ['PassengerId','Name','Ticket','Cabin'],inplace = True)
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [71]:
x_train,x_test,y_train,y_test = train_test_split(df.drop('Survived',axis = 1),df['Survived'], test_size =0.2,random_state= 42)
x_train.shape,x_test.shape

((712, 7), (179, 7))

In [72]:
x_train.head(),y_train.head()  #missing values in embarked and age

(     Pclass     Sex   Age  SibSp  Parch     Fare Embarked
 331       1    male  45.5      0      0  28.5000        S
 733       2    male  23.0      0      0  13.0000        S
 382       3    male  32.0      0      0   7.9250        S
 704       3    male  26.0      1      0   7.8542        S
 813       3  female   6.0      4      2  31.2750        S,
 331    0
 733    0
 382    0
 704    0
 813    0
 Name: Survived, dtype: int64)

In [73]:
# applyin imputation (to solve missing values)
si_age = SimpleImputer()
si_emb = SimpleImputer(strategy ='most_frequent') # for categorical data

x_train_age = si_age.fit_transform(x_train[['Age']])
x_train_emb = si_emb.fit_transform(x_train[['Embarked']])

x_test_age = si_age.transform(x_test[['Age']])
x_test_emb = si_emb.transform(x_test[['Embarked']])

In [74]:
x_train_age.shape,x_train_emb.shape

((712, 1), (712, 1))

In [75]:
# applying ohe (on sex and embarked)
ohe_sex = OneHotEncoder(sparse_output = False, handle_unknown='ignore')
ohe_emb = OneHotEncoder(sparse_output = False, handle_unknown='ignore')

x_train_sex = ohe_sex.fit_transform(x_train[['Sex']])
x_train_emb = ohe_emb.fit_transform(x_train_emb)

x_test_sex = ohe_sex.transform(x_test[['Sex']])
x_test_emb = ohe_emb.transform(x_test_emb)

In [76]:
x_train_sex.shape,x_train_emb.shape

((712, 2), (712, 3))

In [77]:
x_train_sex

array([[0., 1.],
       [0., 1.],
       [0., 1.],
       ...,
       [0., 1.],
       [1., 0.],
       [0., 1.]])

In [78]:
x_train.head(5)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


In [79]:
x_train_rem = x_train.drop(columns = ['Sex','Age','Embarked'])
x_test_rem = x_test.drop(columns = ['Sex','Age','Embarked'])

In [80]:
x_train_rem.shape

(712, 4)

In [81]:
x_train_transform = np.concatenate((x_train_rem,x_train_age,x_train_sex,x_train_emb),axis = 1)
x_test_transform = np.concatenate((x_test_rem,x_test_age,x_test_sex,x_test_emb),axis = 1)

In [82]:
x_train_transform.shape

(712, 10)

In [83]:
clf = DecisionTreeClassifier()
clf.fit(x_train_transform,y_train)

DecisionTreeClassifier()

In [84]:
y_pred = clf.predict(x_test_transform)
y_pred

array([0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1,
       0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1,
       0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0,
       1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0,
       0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0,
       0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0,
       0, 1, 1])

In [85]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.770949720670391

In [93]:
import os
import pickle

In [94]:
os.makedirs('models',exist_ok=True)

pickle.dump(ohe_sex, open("models/ohe_sex.pkl",'wb'))
pickle.dump(ohe_emb, open("models/ohe_emb.pkl",'wb'))
pickle.dump(clf, open("models/clf.pkl",'wb'))

In [96]:
import pickle
import numpy as np

In [99]:
ohe_sex = pickle.load(open('models/ohe_sex.pkl','rb'))
ohe_emb = pickle.load(open('models/ohe_emb.pkl','rb'))
clf = pickle.load(open('models/clf.pkl','rb'))

In [101]:
# assume user input
#Pclass/sex/age/sisp/parch/fare/embarked
test_input = np.array([2,'male',31.0,0,0,10.5,'S'], dtype=object).reshape(1,7)

In [102]:
test_input

array([[2, 'male', 31.0, 0, 0, 10.5, 'S']], dtype=object)

In [103]:
test_input_sex= ohe_sex.transform(test_input[:,1].reshape(1,1))
test_input_emb= ohe_emb.transform(test_input[:,6].reshape(1,1))
test_input_age= test_input[:,2].reshape(1,1)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [104]:
test_input_sex

array([[0., 1.]])

In [106]:
test_input_emb

array([[0., 0., 1.]])

In [107]:
test_input_age

array([[31.0]], dtype=object)

In [111]:
# joining all
test_input_transform = np.concatenate((test_input[:,[0,3,4,5]],test_input_age,test_input_sex,test_input_emb),axis =1)

In [112]:
test_input_transform.shape

(1, 10)

In [113]:
clf.predict(test_input_transform)

array([1])

In [ ]:
# till this we worked manually to predict a random input will survive or not